In [1]:
import polars as pl
from polars import LazyFrame, DataFrame
import numpy as np
from pathlib import Path

## Global settings

In [2]:
merge_threshold = 0.05

data_products_path = Path(r"data_products")
data_products_path.mkdir(parents=True, exist_ok=True)
combined_atlas_path: Path = data_products_path / "combined_atlas.parquet"
combined_mask_path: Path = data_products_path / "combined_mask.parquet"
combined_mask_no_merge_path: Path = data_products_path / "combined_mask_no_merge.parquet"
mask_atlas_combined_path: Path = data_products_path / "mask_atlas_combined.parquet"
boulder_agg_data_path : Path = data_products_path / "boulder_agg_data.parquet"

## Global data

In [3]:
tile_offset_lookup: dict[str, np.ndarray] = { # Like a matrix index (ij)
    "A" : np.array([0, 0], dtype=np.uint32),
    "B" : np.array([0, 512], dtype=np.uint32),
    "C" : np.array([512, 0], dtype=np.uint32),
    "D" : np.array([512, 512], dtype=np.uint32)
}

figures_path = Path("figures")
figures_path.mkdir(parents=True, exist_ok=True)

calculate_tile_pixel_offset = lambda tile_lod_code : sum([(2 ** i) * tile_offset_lookup[t] for i, t in enumerate(tile_lod_code[::-1])]) + np.zeros(2, dtype=np.uint32)

## Step 0 : Merge detections

In [4]:
db_file_path = Path(r"external_data\export_30_detect_threshold_transform.csv")

lf: LazyFrame = pl.scan_csv(db_file_path)

# Add some additional metadata
lf = lf.sort("tile_lod_code") # Very important that row ids are assigned in blacks for tile lod_code for create_merge_db_cache
lf = lf.with_row_index("row_id")
df = lf.collect()

LODS = df["tile_lod_number"].unique().to_list()
MAX_LOD = max(LODS)
num_i = 512 * (2 ** MAX_LOD)
num_j = 512 * (2 ** MAX_LOD)
FACES = df["tile_face"].unique().to_list()

df

row_id,tile_face,tile_lod_number,tile_lod_code,tile_reciprocal_length,tile_reciprocal_area,tile_x_min,tile_x_max,tile_y_min,tile_y_max,relative_bounding_box_x_min,relative_bounding_box_y_min,relative_bounding_box_x_max,relative_bounding_box_y_max,BoulderNet_confidence,detection_index,image_tile_path,LAS_factor_path,image_detections_path
u32,str,i64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,str,str,str
0,"""posx""",0,"""""",1.0,1.0,0.0,1.0,0.0,1.0,234.030762,247.441406,245.602631,260.567596,0.998574,0,"""G:/AO33_pipeline_folders/LAS e…","""G:/AO33_pipeline_folders/LAS e…","""G:/AO33_pipeline_folders/LAS e…"
1,"""posx""",0,"""""",1.0,1.0,0.0,1.0,0.0,1.0,314.020935,361.987518,322.998291,374.02829,0.997989,1,"""G:/AO33_pipeline_folders/LAS e…","""G:/AO33_pipeline_folders/LAS e…","""G:/AO33_pipeline_folders/LAS e…"
2,"""posx""",0,"""""",1.0,1.0,0.0,1.0,0.0,1.0,249.22374,297.653717,256.949432,304.008453,0.996434,2,"""G:/AO33_pipeline_folders/LAS e…","""G:/AO33_pipeline_folders/LAS e…","""G:/AO33_pipeline_folders/LAS e…"
3,"""posx""",0,"""""",1.0,1.0,0.0,1.0,0.0,1.0,395.570221,390.539215,405.14621,398.879608,0.995699,3,"""G:/AO33_pipeline_folders/LAS e…","""G:/AO33_pipeline_folders/LAS e…","""G:/AO33_pipeline_folders/LAS e…"
4,"""posx""",0,"""""",1.0,1.0,0.0,1.0,0.0,1.0,241.668503,118.214088,251.567795,127.494225,0.994353,4,"""G:/AO33_pipeline_folders/LAS e…","""G:/AO33_pipeline_folders/LAS e…","""G:/AO33_pipeline_folders/LAS e…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
3890860,"""negz""",4,"""DDDD""",16.0,256.0,0.9375,1.0,0.9375,1.0,122.00235,293.210571,135.313538,303.805908,0.2177,1083,"""G:/AO33_pipeline_folders/LAS e…","""G:/AO33_pipeline_folders/LAS e…","""G:/AO33_pipeline_folders/LAS e…"
3890861,"""negz""",4,"""DDDD""",16.0,256.0,0.9375,1.0,0.9375,1.0,347.939453,455.238159,358.497375,465.978455,0.214878,1084,"""G:/AO33_pipeline_folders/LAS e…","""G:/AO33_pipeline_folders/LAS e…","""G:/AO33_pipeline_folders/LAS e…"
3890862,"""negz""",4,"""DDDD""",16.0,256.0,0.9375,1.0,0.9375,1.0,79.360153,3.09359,89.768723,12.531713,0.213039,1085,"""G:/AO33_pipeline_folders/LAS e…","""G:/AO33_pipeline_folders/LAS e…","""G:/AO33_pipeline_folders/LAS e…"


In [5]:
from utils.step_0_0 import create_merge_db_cache

cache_folder = Path("refinement_part_0")
merge_db_cache = cache_folder / "merge_db_cache"
merge_db_cache.mkdir(parents=True, exist_ok=True)

if not combined_mask_no_merge_path.exists():

    active_row_ids: pl.DataFrame = create_merge_db_cache(
        df = df,
        faces = FACES,
        lod_levels=LODS,
        cache_folder = merge_db_cache,
        calculate_tile_pixel_offset = calculate_tile_pixel_offset,
        combined_mask_no_merge_path = combined_mask_no_merge_path
    )

In [6]:
print("Finding active row ids...")
active_row_ids: DataFrame = pl.scan_parquet(combined_mask_no_merge_path).group_by("row_id").agg().collect()

df = df.join(active_row_ids, on = "row_id", how = "inner").select(
    "row_id", "tile_face", "tile_lod_number", "tile_lod_code", "tile_reciprocal_length", "tile_reciprocal_area",
    "tile_x_min", "tile_x_max", "tile_y_min", "tile_y_max"
)

df

Finding active row ids...


row_id,tile_face,tile_lod_number,tile_lod_code,tile_reciprocal_length,tile_reciprocal_area,tile_x_min,tile_x_max,tile_y_min,tile_y_max
u32,str,i64,str,f64,f64,f64,f64,f64,f64
1,"""posx""",0,"""""",1.0,1.0,0.0,1.0,0.0,1.0
2,"""posx""",0,"""""",1.0,1.0,0.0,1.0,0.0,1.0
3,"""posx""",0,"""""",1.0,1.0,0.0,1.0,0.0,1.0
4,"""posx""",0,"""""",1.0,1.0,0.0,1.0,0.0,1.0
5,"""posx""",0,"""""",1.0,1.0,0.0,1.0,0.0,1.0
…,…,…,…,…,…,…,…,…,…
3889975,"""negz""",4,"""DDDD""",16.0,256.0,0.9375,1.0,0.9375,1.0
3889976,"""negz""",4,"""DDDD""",16.0,256.0,0.9375,1.0,0.9375,1.0
3889977,"""negz""",4,"""DDDD""",16.0,256.0,0.9375,1.0,0.9375,1.0


In [7]:
from utils.step_0_1 import export_combined_mask_and_merge
merged_df_path: Path = cache_folder / "merged_df.parquet"

if any([not combined_mask_path.exists(), not merged_df_path.exists()]):

    export_combined_mask_and_merge(
        df = df,
        faces = FACES,
        cache_folder = cache_folder,
        combined_mask_path=combined_mask_path,
        merge_threshold=merge_threshold,
        combined_mask_no_merge_path=combined_mask_no_merge_path,
        merged_df_path=merged_df_path
    )

df: DataFrame = pl.read_parquet(merged_df_path)

## Step 1 : Create attribute atlas

In [8]:
from utils.step_1 import create_combined_atlas_components

create_combined_atlas_components(
    df = df,
    faces = FACES,
    max_lod_level = MAX_LOD,
    calculate_tile_pixel_offset = calculate_tile_pixel_offset,
    attribute_atlas_cache_folder = Path(r"refinement_part_1")
)

Processing face posy with lod depth 4: 100%|██████████| 256/256 [00:00<00:00, 2157.96it/s]


In [9]:
from utils.step_1 import merge_combined_atlas_components

if not combined_atlas_path.exists():

    merge_combined_atlas_components(
        combined_atlas_path = combined_atlas_path
    )

## Step 2 : Fill combined merge with the relevant atlas data

In [10]:
from utils.step_2 import join_mask_and_atlas_tables

if not mask_atlas_combined_path.exists():

    join_mask_and_atlas_tables(
        faces = FACES,
        combined_mask_path=combined_mask_path,
        combined_atlas_path=combined_atlas_path,
        mask_atlas_combined_path=mask_atlas_combined_path
    )

## Step 3.0 : Load generated LazyFrames

In [11]:
combined_mask: LazyFrame = pl.scan_parquet(combined_mask_path)
combined_atlas: LazyFrame = pl.scan_parquet(combined_atlas_path)
mask_atlas_combined: LazyFrame = pl.scan_parquet(mask_atlas_combined_path)

print(f"combined_mask : has schema {combined_mask.collect_schema()} and additional information {combined_mask.explain()}")
print(f"combined_atlas : has schema {combined_atlas.collect_schema()} and additional information {combined_atlas.explain()}")
print(f"mask_atlas_combined : has schema {mask_atlas_combined.collect_schema()} and additional information {mask_atlas_combined.explain()}")

combined_mask : has schema Schema({'lod_level': UInt8, 'lod_code': String, 'face': String, 'i': UInt32, 'j': UInt32, 'row_id': UInt32}) and additional information Parquet SCAN [data_products\combined_mask.parquet]
PROJECT */6 COLUMNS
ESTIMATED ROWS: 134519709
combined_atlas : has schema Schema({'face': String, 'i': UInt32, 'j': UInt32, 'uint8_reflectance': UInt8, '32bit_reflectance': Float32, 'positions_x': Float32, 'positions_y': Float32, 'positions_z': Float32}) and additional information Parquet SCAN [data_products\combined_atlas.parquet]
PROJECT */8 COLUMNS
ESTIMATED ROWS: 402653184
mask_atlas_combined : has schema Schema({'lod_level': UInt8, 'lod_code': String, 'face': String, 'i': UInt32, 'j': UInt32, 'row_id': UInt32, 'uint8_reflectance': UInt8, '32bit_reflectance': Float32, 'positions_x': Float32, 'positions_y': Float32, 'positions_z': Float32}) and additional information Parquet SCAN [data_products\mask_atlas_combined.parquet]
PROJECT */11 COLUMNS
ESTIMATED ROWS: 134519709


## Step 3.1 : Finding longest axis for each boulder in meters

In [12]:
df = pl.read_parquet(boulder_agg_data_path) if boulder_agg_data_path.exists() else \
    df.select(["row_id", "tile_face", "tile_lod_number", "tile_lod_code"])

In [13]:
from utils.step_3_1 import get_longest_axis_diameter_lookup

if "longest_axis_diameter" not in df.columns:

    longest_axis_diameters, surface_areas = get_longest_axis_diameter_lookup(mask_atlas_combined)

    df = df.with_columns(
        pl.col("row_id")
        .replace_strict(longest_axis_diameters, default=None)
        .cast(pl.Float32)
        .alias("longest_axis_diameter"),

        pl.col("row_id")
        .replace_strict(surface_areas, default=None)
        .cast(pl.Float32)
        .alias("surface_area")
    )

    df.write_parquet(boulder_agg_data_path)

## Step 3.2 : Aggregate data

In [14]:
from utils.step_3_2 import agg_stats

if "mean_i" not in df.columns:

    agg_df = agg_stats(mask_atlas_combined)

    df = df.join(
        agg_df,
            on=["row_id"],
            how="inner"
        )
    
    df.write_parquet(boulder_agg_data_path)

## Step 3.3 : Find LAS

In [15]:
from utils.step_3_3 import find_LAS

cache_folder = Path("refinement_part_3_3")
cache_folder.mkdir(exist_ok=True)

Phi_export_path_mesh: Path = cache_folder / "Phi_export_mesh"
Phi_export_path_sphere: Path = cache_folder / "Phi_export_sphere"

find_LAS(
    cache_folder=cache_folder,
    faces=FACES,
    num_i=num_i,
    combined_atlas=combined_atlas,
    Phi_export_path_mesh=Phi_export_path_mesh,
    Phi_export_path_sphere=Phi_export_path_sphere
)

Running steps for face posy: 100%|██████████| 8/8 [00:00<00:00, 2338.61it/s]


## Step 3.4 : Find Phi Counts

In [16]:
from pathlib import Path
from utils.step_3_4 import run_compute_Phi_counts

phi_counts_cache_folder = Path("refinement_part_3_4")

Phi_counts_smoothed_path: Path = phi_counts_cache_folder / "Phi_counts_smoothed"
Phi_counts_noisy_path: Path = phi_counts_cache_folder / "Phi_counts_noisy"
Phi_counts_sphere_theoretical_path : Path = phi_counts_cache_folder / "Phi_counts_sphere_theoretical"
Phi_counts_sphere_noisy_path : Path = phi_counts_cache_folder / "Phi_counts_sphere_noisy"

run_compute_Phi_counts(
        Phi_counts_smoothed_path, Phi_counts_noisy_path,
        Phi_counts_sphere_theoretical_path, Phi_counts_sphere_noisy_path,

        Phi_mesh = pl.scan_parquet(Phi_export_path_mesh),
        Phi_sphere = pl.scan_parquet(Phi_export_path_sphere),
        bin_numbers = np.array([128, 256, 512, 1024, 2048, 4096])
)

Finding min and max for Phi_counts_smoothed:   0%|          | 0/64 [00:00<?, ?it/s]

Populating bins for Phi_counts_noisy: 100%|██████████| 64/64 [02:15<00:00,  2.12s/it]
Finding min and max for Phi_counts_sphere_theoretical: 100%|██████████| 64/64 [01:59<00:00,  1.86s/it]
Populating bins for Phi_counts_sphere_theoretical: 100%|██████████| 64/64 [02:07<00:00,  1.99s/it]
Populating bins for Phi_counts_sphere_noisy: 100%|██████████| 64/64 [02:16<00:00,  2.14s/it]
